# Olist 数据质量检查

本 Notebook 用于：
1. 检查数据完整性（缺失值、重复值）
2. 验证数据一致性（主外键关系）
3. 验证指标口径（item_gmv vs paid_gmv）
4. 生成数据质量报告

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # macOS 中文支持
plt.rcParams['axes.unicode_minus'] = False

## 1. 加载数据

In [3]:
# 数据路径
data_path = Path('C:\\Users\\21042\\Downloads\\archive')

# 加载主要数据表
orders = pd.read_csv(data_path / 'olist_orders_dataset.csv')
order_items = pd.read_csv(data_path / 'olist_order_items_dataset.csv')
order_payments = pd.read_csv(data_path / 'olist_order_payments_dataset.csv')
customers = pd.read_csv(data_path / 'olist_customers_dataset.csv')
reviews = pd.read_csv(data_path / 'olist_order_reviews_dataset.csv')
products = pd.read_csv(data_path / 'olist_products_dataset.csv')

print(f"订单表: {orders.shape}")
print(f"订单明细表: {order_items.shape}")
print(f"支付表: {order_payments.shape}")
print(f"客户表: {customers.shape}")
print(f"评价表: {reviews.shape}")
print(f"商品表: {products.shape}")

订单表: (99441, 8)
订单明细表: (112650, 7)
支付表: (103886, 5)
客户表: (99441, 5)
评价表: (99224, 7)
商品表: (32951, 9)


## 2. 数据完整性检查

In [4]:
def check_missing_values(df, table_name):
    """检查缺失值"""
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        '缺失数': missing,
        '缺失率(%)': missing_pct
    })
    missing_df = missing_df[missing_df['缺失数'] > 0].sort_values('缺失数', ascending=False)
    
    print(f"\n=== {table_name} 缺失值检查 ===")
    if len(missing_df) > 0:
        print(missing_df)
    else:
        print("✓ 无缺失值")
    return missing_df

# 检查各表缺失值
check_missing_values(orders, '订单表')
check_missing_values(order_items, '订单明细表')
check_missing_values(order_payments, '支付表')
check_missing_values(reviews, '评价表')


=== 订单表 缺失值检查 ===
                                缺失数  缺失率(%)
order_delivered_customer_date  2965    2.98
order_delivered_carrier_date   1783    1.79
order_approved_at               160    0.16

=== 订单明细表 缺失值检查 ===
✓ 无缺失值

=== 支付表 缺失值检查 ===
✓ 无缺失值

=== 评价表 缺失值检查 ===
                          缺失数  缺失率(%)
review_comment_title    87656   88.34
review_comment_message  58247   58.70


,缺失数,缺失率(%)
review_comment_title,87656,88.34
review_comment_message,58247,58.70


## 3. 数据一致性检查

In [5]:
# 检查订单状态分布
print("\n=== 订单状态分布 ===")
print(orders['order_status'].value_counts())
print(f"\n已送达订单占比: {(orders['order_status'] == 'delivered').sum() / len(orders) * 100:.2f}%")

# 检查主外键关系
print("\n=== 主外键一致性检查 ===")
print(f"订单表订单数: {orders['order_id'].nunique()}")
print(f"订单明细表订单数: {order_items['order_id'].nunique()}")
print(f"支付表订单数: {order_payments['order_id'].nunique()}")
print(f"评价表订单数: {reviews['order_id'].nunique()}")

# 检查是否有订单明细但订单表中不存在的情况
orphan_items = set(order_items['order_id']) - set(orders['order_id'])
print(f"\n孤立订单明细数: {len(orphan_items)}")


=== 订单状态分布 ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

已送达订单占比: 97.02%

=== 主外键一致性检查 ===
订单表订单数: 99441
订单明细表订单数: 98666
支付表订单数: 99440
评价表订单数: 98673

孤立订单明细数: 0


## 4. 指标口径验证

In [6]:
# 仅统计已送达订单
delivered_orders = orders[orders['order_status'] == 'delivered']['order_id']

# 计算 item_gmv（订单明细口径）
item_gmv_df = order_items[order_items['order_id'].isin(delivered_orders)].copy()
item_gmv_df['item_amount'] = item_gmv_df['price'] + item_gmv_df['freight_value']
item_gmv = item_gmv_df.groupby('order_id')['item_amount'].sum().reset_index()
item_gmv.columns = ['order_id', 'item_gmv']

# 计算 paid_gmv（支付口径）
paid_gmv = order_payments[order_payments['order_id'].isin(delivered_orders)].groupby('order_id')['payment_value'].sum().reset_index()
paid_gmv.columns = ['order_id', 'paid_gmv']

# 合并对比
gmv_compare = item_gmv.merge(paid_gmv, on='order_id', how='outer')
gmv_compare['diff'] = gmv_compare['item_gmv'] - gmv_compare['paid_gmv']
gmv_compare['diff_pct'] = (gmv_compare['diff'] / gmv_compare['item_gmv'] * 100).round(2)

print("\n=== GMV 口径对比 ===")
print(f"item_gmv 总计: R$ {gmv_compare['item_gmv'].sum():,.2f}")
print(f"paid_gmv 总计: R$ {gmv_compare['paid_gmv'].sum():,.2f}")
print(f"差异: R$ {gmv_compare['diff'].sum():,.2f}")
print(f"差异率: {(gmv_compare['diff'].sum() / gmv_compare['item_gmv'].sum() * 100):.2f}%")

# 查看差异分布
print("\n差异分布统计:")
print(gmv_compare['diff'].describe())


=== GMV 口径对比 ===
item_gmv 总计: R$ 15,419,773.75
paid_gmv 总计: R$ 15,422,461.77
差异: R$ -2,831.48
差异率: -0.02%

差异分布统计:
count    96477.000000
mean        -0.029349
std          1.138706
min       -182.810000
25%          0.000000
50%          0.000000
75%          0.000000
max         51.620000
Name: diff, dtype: float64


## 5. 数据质量报告总结

In [7]:
print("\n" + "="*50)
print("数据质量检查总结")
print("="*50)
print(f"✓ 总订单数: {len(orders):,}")
print(f"✓ 已送达订单数: {len(delivered_orders):,} ({len(delivered_orders)/len(orders)*100:.1f}%)")
print(f"✓ 订单明细行数: {len(order_items):,}")
print(f"✓ 支付记录数: {len(order_payments):,}")
print(f"✓ 评价记录数: {len(reviews):,}")
print(f"\n✓ item_gmv 与 paid_gmv 差异率: {(gmv_compare['diff'].sum() / gmv_compare['item_gmv'].sum() * 100):.2f}%")
print("\n数据质量: 良好，可用于后续分析")


数据质量检查总结
✓ 总订单数: 99,441
✓ 已送达订单数: 96,478 (97.0%)
✓ 订单明细行数: 112,650
✓ 支付记录数: 103,886
✓ 评价记录数: 99,224

✓ item_gmv 与 paid_gmv 差异率: -0.02%

数据质量: 良好，可用于后续分析
